In [8]:
import spacy
import torch
import torch.nn as nn
import torch.optim as optim

nlp = spacy.load("en_core_web_sm")

corpus = """
Neural networks are useful for deep learning.
Deep learning uses neural networks for many applications.
Machine learning and deep learning are important technologies.
"""

doc = nlp(corpus.lower())
words = [token.text for token in doc if token.is_alpha]

vocab = sorted(set(words))
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

window_size = 2
training_data = []

for i, word in enumerate(words):
    center_word = word_to_idx[word]
    for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
        if i != j:
            context_word = word_to_idx[words[j]]
            training_data.append((center_word, context_word))

class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.linear(x)
        return x

vocab_size = len(vocab)
embedding_dim = 50

model = Word2Vec(vocab_size, embedding_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    total_loss = 0

    for center_word, context_word in training_data:
        center_tensor = torch.tensor([center_word], dtype=torch.long)
        context_tensor = torch.tensor([context_word], dtype=torch.long)

        optimizer.zero_grad()

        output = model(center_tensor)
        loss = criterion(output, context_tensor)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")

print("\nWord Embeddings:")

for word in vocab:
    word_index = word_to_idx[word]
    embedding = model.embedding.weight[word_index].detach().numpy()
    print(word, embedding)

Epoch 10, Loss: 179.0328
Epoch 20, Loss: 172.6920
Epoch 30, Loss: 169.2066
Epoch 40, Loss: 167.0095
Epoch 50, Loss: 165.6646
Epoch 60, Loss: 164.4178
Epoch 70, Loss: 163.4875
Epoch 80, Loss: 162.8258
Epoch 90, Loss: 162.0741
Epoch 100, Loss: 161.5492

Word Embeddings:
and [ 0.4450975   0.9933015   0.37423754 -0.34376344 -0.6755625   0.27624258
 -0.20654547 -0.20520544  0.2976168   0.26257634 -1.4915099   0.12514491
 -0.17735662 -0.9050554  -1.1981126   0.8627454  -1.9961876   0.7376837
  0.6104696  -0.3406352   0.54180926  0.26791993  0.22247007  0.9974641
  0.4248379  -1.0181866   1.1141638  -0.54214686  0.21230754  1.1578106
  0.01732403 -0.01536742 -0.76265144  0.08125508  0.03639759 -0.24141644
  0.69639754 -0.957198   -1.153795   -1.2607077  -0.44361266 -0.01189279
  1.0131558  -0.03145886 -0.0571024   0.9810556  -0.24670228  0.26111823
 -0.28344336 -0.02185911]
applications [-0.19805656  0.42774248 -0.4137588   1.0652816   0.39121294  0.2668842
 -0.5362773   0.02695028 -0.1459105

In [7]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ----- ---------------------------------- 1.8/12.8 MB 10.2 MB/s eta 0:00:02
     ------------- -------------------------- 4.2/12.8 MB 11.1 MB/s eta 0:00:01
     -------------------- ------------------- 6.6/12.8 MB 11.3 MB/s eta 0:00:01
     --------------------------- ------------ 8.9/12.8 MB 11.4 MB/s eta 0:00:01
     ---------------------------------- ---- 11.3/12.8 MB 11.5 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 11.3 MB/s  0:00:01
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
